# EDA — Online Retail II
## Segmentation client par analyse RFM

**Dataset** : UCI Online Retail II  
**Source** : https://archive.uci.edu/dataset/502/online+retail+ii  
**Objectif** : Explorer les données de transactions e-commerce et préparer les features RFM pour le clustering client.

## 0. Imports et configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Style global
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print('Imports OK')

## 1. Chargement des données

Le dataset contient deux feuilles Excel (2009-2010 et 2010-2011).  
On les concatène pour avoir l'ensemble des transactions.

In [ ]:
# Chemin vers le fichier (à placer dans data/)
DATA_PATH = '../data/online_retail_II.xlsx'

# Chargement des deux feuilles
df1 = pd.read_excel(DATA_PATH, sheet_name='Year 2009-2010', engine='openpyxl')
df2 = pd.read_excel(DATA_PATH, sheet_name='Year 2010-2011', engine='openpyxl')

# Concaténation
df = pd.concat([df1, df2], ignore_index=True)

print(f'Dimensions totales : {df.shape}')
print(f'Période : {df["InvoiceDate"].min()} → {df["InvoiceDate"].max()}')
df.head()

## 2. Exploration initiale

In [ ]:
# Types et aperçu général
df.info()

In [ ]:
# Statistiques descriptives
df.describe()

In [ ]:
# Valeurs manquantes
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Valeurs manquantes': missing,
    'Pourcentage (%)': missing_pct
})

print(missing_df[missing_df['Valeurs manquantes'] > 0])

**Observation** : La colonne `Customer ID` contient ~25% de valeurs manquantes.  
Ces lignes correspondent à des achats sans compte client (guests) — elles seront exclues pour le clustering RFM car on ne peut pas les suivre dans le temps.

## 3. Nettoyage préliminaire

In [ ]:
print(f'Lignes avant nettoyage : {len(df):,}')

# 1. Supprimer les lignes sans Customer ID
df = df.dropna(subset=['Customer ID'])
print(f'Après suppression Customer ID nuls : {len(df):,}')

# 2. Supprimer les transactions annulées (Invoice commençant par C)
df = df[~df['Invoice'].astype(str).str.startswith('C')]
print(f'Après suppression annulations : {len(df):,}')

# 3. Supprimer les quantités et prix négatifs ou nuls
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]
print(f'Après suppression quantités/prix invalides : {len(df):,}')

# 4. Créer la colonne TotalPrice
df['TotalPrice'] = df['Quantity'] * df['Price']

# 5. S'assurer que Customer ID est bien en entier
df['Customer ID'] = df['Customer ID'].astype(int)

print(f'\nDataset nettoyé : {df.shape}')

## 4. Distribution géographique

In [ ]:
# Top 10 pays par nombre de transactions
top_countries = df['Country'].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : nombre de transactions
top_countries.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Top 10 pays — Nombre de transactions', fontweight='bold')
axes[0].set_xlabel('Pays')
axes[0].set_ylabel('Nombre de transactions')
axes[0].tick_params(axis='x', rotation=45)

# Graphique 2 : CA par pays
top_ca = df.groupby('Country')['TotalPrice'].sum().sort_values(ascending=False).head(10)
top_ca.plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Top 10 pays — Chiffre d\'affaires (£)', fontweight='bold')
axes[1].set_xlabel('Pays')
axes[1].set_ylabel('CA total (£)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../data/viz_countries.png', bbox_inches='tight')
plt.show()

uk_pct = (df['Country'].value_counts()['United Kingdom'] / len(df) * 100).round(1)
print(f'\nLe Royaume-Uni représente {uk_pct}% des transactions.')

## 5. Évolution temporelle des ventes

In [ ]:
# CA mensuel
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')
monthly_revenue = df.groupby('YearMonth')['TotalPrice'].sum()

fig, ax = plt.subplots(figsize=(14, 5))
monthly_revenue.plot(kind='line', ax=ax, color='steelblue', linewidth=2, marker='o', markersize=4)
ax.set_title('Évolution mensuelle du chiffre d\'affaires', fontweight='bold')
ax.set_xlabel('Mois')
ax.set_ylabel('CA total (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../data/viz_monthly_revenue.png', bbox_inches='tight')
plt.show()

print('Observation : pic de ventes visible en novembre (préparation Noël).')

## 6. Distribution des montants par transaction

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution brute (avec outliers)
axes[0].hist(df['TotalPrice'], bins=100, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution TotalPrice (brute)', fontweight='bold')
axes[0].set_xlabel('Montant (£)')
axes[0].set_ylabel('Fréquence')

# Distribution filtrée (sans outliers extrêmes)
df_filtered = df[df['TotalPrice'] < df['TotalPrice'].quantile(0.99)]
axes[1].hist(df_filtered['TotalPrice'], bins=100, color='coral', edgecolor='white')
axes[1].set_title('Distribution TotalPrice (99e percentile)', fontweight='bold')
axes[1].set_xlabel('Montant (£)')
axes[1].set_ylabel('Fréquence')

plt.tight_layout()
plt.savefig('../data/viz_price_distribution.png', bbox_inches='tight')
plt.show()

print(f'Médiane : £{df["TotalPrice"].median():.2f}')
print(f'Moyenne : £{df["TotalPrice"].mean():.2f}')
print(f'Max : £{df["TotalPrice"].max():,.2f}')

## 7. Calcul des features RFM

In [ ]:
# Date de référence = lendemain du dernier achat
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f'Date de référence : {reference_date.date()}')

# Calcul RFM par client
rfm = df.groupby('Customer ID').agg(
    Recency   = ('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency = ('Invoice',     'nunique'),
    Monetary  = ('TotalPrice',  'sum')
).reset_index()

print(f'\nNombre de clients uniques : {len(rfm):,}')
rfm.head(10)

In [ ]:
# Statistiques RFM
rfm[['Recency', 'Frequency', 'Monetary']].describe().round(2)

## 8. Visualisation des distributions RFM

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Recency
axes[0].hist(rfm['Recency'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution — Recency (jours)', fontweight='bold')
axes[0].set_xlabel('Jours depuis dernier achat')
axes[0].set_ylabel('Nombre de clients')

# Frequency
rfm_freq = rfm[rfm['Frequency'] < rfm['Frequency'].quantile(0.95)]
axes[1].hist(rfm_freq['Frequency'], bins=50, color='coral', edgecolor='white')
axes[1].set_title('Distribution — Frequency (commandes)', fontweight='bold')
axes[1].set_xlabel('Nombre de commandes')
axes[1].set_ylabel('Nombre de clients')

# Monetary
rfm_mon = rfm[rfm['Monetary'] < rfm['Monetary'].quantile(0.95)]
axes[2].hist(rfm_mon['Monetary'], bins=50, color='mediumseagreen', edgecolor='white')
axes[2].set_title('Distribution — Monetary (£)', fontweight='bold')
axes[2].set_xlabel('CA total par client (£)')
axes[2].set_ylabel('Nombre de clients')

plt.tight_layout()
plt.savefig('../data/viz_rfm_distributions.png', bbox_inches='tight')
plt.show()

## 9. Corrélations entre variables RFM

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
corr = rfm[['Recency', 'Frequency', 'Monetary']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Corrélations RFM', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/viz_rfm_correlation.png', bbox_inches='tight')
plt.show()

## 10. Top clients et produits

In [ ]:
# Top 10 clients par CA
top_clients = rfm.nlargest(10, 'Monetary')[['Customer ID', 'Recency', 'Frequency', 'Monetary']]
print('=== Top 10 clients par CA ===')
print(top_clients.to_string(index=False))

In [ ]:
# Top 10 produits par CA
top_products = (
    df.groupby('Description')['TotalPrice']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(12, 5))
top_products.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 10 produits par chiffre d\'affaires', fontweight='bold')
ax.set_xlabel('CA total (£)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../data/viz_top_products.png', bbox_inches='tight')
plt.show()

## 11. Synthèse — Points clés de l'EDA

| Observation | Implication pour le projet |
|-------------|---------------------------|
| ~25% de Customer ID manquants | Exclusion des guests, focus sur clients enregistrés |
| Forte concentration UK (~90%) | Biais géographique à mentionner |
| Pic de ventes en novembre | Saisonnalité à prendre en compte |
| Distributions RFM très asymétriques | Normalisation / log-transform nécessaire avant clustering |
| Frequency et Monetary corrélés positivement | Cohérent : les clients fréquents dépensent plus |
| Présence d'outliers importants | Traitement nécessaire (winsorization ou log-scale) |

## Prochaines étapes (Assignment 2)

- Normalisation des features RFM (StandardScaler ou log + StandardScaler)
- Traitement des outliers
- Préparation du dataset final pour le clustering